# @examples/pvlib-python

A set of documented functions for simulating the performance of photovoltaic energy systems.

Published version **v1.0.0** — this notebook fetches that exact snapshot from the
PowerAI Hub, not the upstream repository's latest commit.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9).

The Hub does not run primitives. This notebook runs in *your* Colab session, on
Google's hardware, under your account.

[View on the Hub](https://hub.powerai.ai/examples/pvlib-python)


In [ ]:
PRIMITIVE = "@examples/pvlib-python"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/examples/pvlib-python/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import subprocess, sys
from pathlib import Path

req = next(Path(WORKDIR).rglob("requirements.txt"), None)
if req is None:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
else:
    print(f"Installing {req.relative_to(WORKDIR)} …")
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        capture_output=True, text=True,
    )
    print("done" if done.returncode == 0 else f"pip failed:\n{done.stderr[-2000:]}")


In [ ]:
import os
os.chdir(WORKDIR)

readme = next((p for p in sorted(Path(WORKDIR).rglob("README*")) if p.is_file()), None)
print(readme.read_text(errors="replace")[:3000] if readme else "No README in this primitive.")


## Serve this primitive as an endpoint

The cells below install the archive you just unpacked, start the endpoint that answers
this primitive's declared tool interface, and open a public HTTPS tunnel to it — so the
Hub, an agent, or a `curl` can call it and get a real computed answer.

Two things to know before you rely on it:

* **It dies with this session.** Colab reclaims idle notebooks after ~90 minutes and
  caps sessions around 12 hours. The Hub stores one address per primitive, so when this
  session ends that address answers nothing until you re-run and paste a new one.
* **Each person who runs this gets a different URL.** One Hub, one address — so this is
  a demo you drive, not something visitors wire up themselves.

The next cell downloads `cloudflared` (Cloudflare's official release), because a Colab
VM has no inbound networking of its own.


In [ ]:
import importlib, subprocess, sys
from pathlib import Path

MODULE = "pvlib"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]", "pandas"], check=True)

# The archive is imported rather than installed, so nothing resolves its dependencies
# for us. They are declared in pyproject (pvlib has no requirements.txt, which is why
# the generic cell above found nothing to do), and reading them costs no build.
def _declared_deps(root):
    pyproject = Path(root) / "pyproject.toml"
    if pyproject.is_file():
        import tomllib

        table = tomllib.loads(pyproject.read_text())
        return list(table.get("project", {}).get("dependencies", []))
    requirements = Path(root) / "requirements.txt"
    if requirements.is_file():
        return [
            line.strip()
            for line in requirements.read_text().splitlines()
            if line.strip() and not line.startswith("#")
        ]
    return []


deps = _declared_deps(WORKDIR)
if deps:
    print(f"installing {len(deps)} declared dependencies …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)

# Import the archive off the path rather than pip-installing it. A published archive is
# a source tree with no .git, and a package whose version comes from setuptools_scm —
# pvlib does — fails at "Getting requirements to build wheel" for exactly that reason.
# Adding it to sys.path needs no build at all, and runs the published bytes rather than
# a rebuild of them.
sys.path.insert(0, WORKDIR)
for stale in [m for m in sys.modules if m == MODULE or m.startswith(MODULE + ".")]:
    del sys.modules[stale]

SERVING = ""
try:
    loaded = importlib.import_module(MODULE)
    # Colab preinstalls plenty; make sure this is the archive's copy and not one of
    # those, or "serving the published version" would be a lie.
    if WORKDIR in str(getattr(loaded, "__file__", "")):
        SERVING = "archive"
        print(f"serving the published version from {WORKDIR}")
    else:
        print(f"{MODULE} resolved to {loaded.__file__}, not the archive")
except Exception as exc:
    print("archive is not importable as-is:", exc)

if SERVING != "archive":
    print("falling back to PyPI pvlib — NOTE: no longer the published version")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pvlib"], check=True)
    SERVING = "pypi"

TOOL_NAME = "clearsky_irradiance"

# Read from the checked-in service at generation time, so the notebook and the
# reference endpoint cannot drift apart.
Path("/content/endpoint.py").write_text('"""Reference Endpoint for the PowerAI Hub primitive @examples/pvlib-python.\n\nAnswers the `clearsky_irradiance` tool interface that primitive declares, by\ncalling pvlib for real. Its only purpose is to show what an author hosts: the Hub\npublishes this address and never calls it, so nothing in the catalog may depend on\nthis process running.\n\n    uv run uvicorn main:app --port 8011\n\nUntrusted input crosses this boundary — anyone who can reach the URL can post to\nit — so every field is range-checked by pydantic before pvlib sees it, and the\nresponse carries no detail about this host.\n"""\n\nfrom __future__ import annotations\n\nfrom datetime import date as Date\nfrom typing import Annotated, Literal\n\nimport pandas as pd\nfrom fastapi import FastAPI, Request\nfrom fastapi.responses import JSONResponse\nfrom pvlib.location import Location\nfrom pydantic import BaseModel, Field, ValidationError\n\napp = FastAPI(title="pvlib clear-sky endpoint", version="0.1.0")\n\n#: Protocol version, echoed on every reply. See docs/endpoint-protocol.md.\nPROTOCOL = "1"\n\n#: 15-minute steps: fine enough that the peak is not missed by much, coarse\n#: enough that a day is 96 rows. Each sample therefore covers a quarter hour,\n#: which is what turns a W/m² series into Wh/m².\nSAMPLE_FREQ = "15min"\nHOURS_PER_SAMPLE = 0.25\n\n\nclass ClearskyRequest(BaseModel):\n    latitude: Annotated[float, Field(ge=-90, le=90, description="Degrees north")]\n    longitude: Annotated[float, Field(ge=-180, le=180, description="Degrees east")]\n    date: Annotated[Date, Field(description="UTC day, YYYY-MM-DD")]\n    # pvlib also offers \'haurwitz\', but it returns GHI *only* — no DNI or DHI. It is\n    # excluded rather than filled with nulls, because the tool interface this\n    # endpoint answers declares all three, and a contract that is true for some\n    # inputs is not true.\n    model: Literal["ineichen", "simplified_solis"] = "ineichen"\n\n\nclass ClearskyResponse(BaseModel):\n    ghi_peak_w_m2: float\n    dni_peak_w_m2: float\n    dhi_peak_w_m2: float\n    ghi_daily_wh_m2: float\n    model: str\n\n\ndef clearsky(req: ClearskyRequest) -> ClearskyResponse:\n    """Peak and daily-total clear-sky irradiance for one UTC day at one place."""\n    location = Location(req.latitude, req.longitude, tz="UTC")\n    times = pd.date_range(\n        start=f"{req.date} 00:00", end=f"{req.date} 23:59", freq=SAMPLE_FREQ, tz="UTC"\n    )\n    frame = location.get_clearsky(times, model=req.model)\n    return ClearskyResponse(\n        ghi_peak_w_m2=round(float(frame["ghi"].max()), 1),\n        dni_peak_w_m2=round(float(frame["dni"].max()), 1),\n        dhi_peak_w_m2=round(float(frame["dhi"].max()), 1),\n        ghi_daily_wh_m2=round(float(frame["ghi"].sum()) * HOURS_PER_SAMPLE, 1),\n        model=req.model,\n    )\n\n\ndef _failure(code: str, message: str, status: int) -> JSONResponse:\n    """A failure the caller can act on, legible without the status code.\n\n    `ok` is the load-bearing field: Alfred hands the model the response *body as a\n    string*, so an agent never sees the HTTP status. Without `ok` a well-formed\n    error is indistinguishable from a result, and the agent reports a failure as an\n    answer.\n    """\n    return JSONResponse(\n        {"powerai": PROTOCOL, "ok": False, "error": {"code": code, "message": message}},\n        status_code=status,\n    )\n\n\n@app.post("/clearsky")\nasync def post_clearsky(request: Request) -> JSONResponse:\n    """Answer either the enveloped or the bare form.\n\n    Enveloped is the documented protocol; bare arguments are what this endpoint\n    accepted before it existed. Detecting the envelope by its `powerai` key keeps\n    anything already built against the old shape working.\n    """\n    try:\n        body = await request.json()\n    except Exception:  # noqa: BLE001 — malformed JSON is the caller\'s mistake, not a crash\n        return _failure("invalid_input", "Request body is not valid JSON.", 400)\n    if not isinstance(body, dict):\n        return _failure("invalid_input", "Request body must be a JSON object.", 400)\n\n    arguments = body.get("input") if body.get("powerai") else body\n    if not isinstance(arguments, dict):\n        return _failure("invalid_input", "`input` must be a JSON object.", 400)\n\n    try:\n        parsed = ClearskyRequest.model_validate(arguments)\n    except ValidationError as exc:\n        first = exc.errors()[0]\n        field = ".".join(str(p) for p in first.get("loc", ())) or "input"\n        return _failure("invalid_input", f"{field}: {first.get(\'msg\', \'invalid\')}", 400)\n\n    try:\n        result = clearsky(parsed)\n    except Exception:  # noqa: BLE001 — never leak a stack trace or host detail\n        return _failure("internal", "The calculation failed.", 500)\n\n    return JSONResponse({"powerai": PROTOCOL, "ok": True, "output": result.model_dump()})\n\n\n@app.get("/health")\ndef health() -> dict[str, str]:\n    return {"status": "ok"}\n\n\ndef _self_check() -> None:\n    """Assert the physics is the right order of magnitude, not just that it runs.\n\n    A bad unit conversion or a mixed-up column still returns a plausible-looking\n    float, so the checks below pin behaviour that only holds if the calculation is\n    actually right: summer beats winter, the equator beats the Arctic in December,\n    and the daily total is consistent with the peak.\n    """\n    summer = clearsky(ClearskyRequest(latitude=37.0, longitude=-122.0, date=Date(2026, 6, 21)))\n    winter = clearsky(ClearskyRequest(latitude=37.0, longitude=-122.0, date=Date(2026, 12, 21)))\n\n    # Clear-sky GHI at temperate latitudes peaks near ~1000 W/m² in midsummer.\n    assert 700 < summer.ghi_peak_w_m2 < 1200, summer\n    assert summer.ghi_peak_w_m2 > winter.ghi_peak_w_m2, (summer, winter)\n    assert summer.ghi_daily_wh_m2 > winter.ghi_daily_wh_m2, (summer, winter)\n\n    # A day cannot deliver more than peak × 24 h, and a sunny summer day well\n    # exceeds one peak-hour — this is what catches a broken Wh conversion.\n    assert summer.ghi_peak_w_m2 < summer.ghi_daily_wh_m2 < summer.ghi_peak_w_m2 * 24, summer\n\n    # Polar night: the sun does not rise, so there is no irradiance at all.\n    polar = clearsky(ClearskyRequest(latitude=78.0, longitude=15.0, date=Date(2026, 12, 21)))\n    assert polar.ghi_peak_w_m2 == 0.0, polar\n\n    # The model argument must actually change the answer, and every accepted model\n    # must fill the whole declared response — the reason \'haurwitz\' is not offered.\n    solis = clearsky(\n        ClearskyRequest(\n            latitude=37.0, longitude=-122.0, date=Date(2026, 6, 21), model="simplified_solis"\n        )\n    )\n    assert solis.ghi_peak_w_m2 != summer.ghi_peak_w_m2, (solis, summer)\n    assert solis.dni_peak_w_m2 > 0 and solis.dhi_peak_w_m2 > 0, solis\n\n    print(f"summer  {summer.model_dump()}")\n    print(f"winter  {winter.model_dump()}")\n    print(f"polar   {polar.model_dump()}")\n    print(f"solis   {solis.model_dump()}")\n    print("self-check OK")\n\n\nif __name__ == "__main__":\n    _self_check()\n')
print("wrapper written to /content/endpoint.py")


In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8011
ROUTE = "/clearsky"
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe, and without this it is not. A previous run
# leaves a tunnel and a server alive, so the next one would: fail to overwrite the
# cloudflared binary while it is executing ("Text file busy"), fail to bind the port,
# and then pass the health probe anyway — because the *old* server answered it. That
# last part is the dangerous one: a stale process serving a stale archive, reported as
# a fresh success.
for pattern in ("cloudflared tunnel", "uvicorn endpoint:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    # Colab VMs have no inbound networking, so a tunnel is the only way in. Quiet
    # unless it fails, and then loud: `check=True` alone raises with no detail.
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# sys.path does not cross into a child process, so the archive has to be handed over
# explicitly — otherwise this server imports whatever is installed globally while the
# cell above reports it is serving the published version.
env = dict(os.environ)
if SERVING == "archive":
    env["PYTHONPATH"] = WORKDIR + os.pathsep + env.get("PYTHONPATH", "")

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "endpoint:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/content", env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait for the app before exposing it, so the tunnel never fronts a dead port.
for _ in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("the endpoint did not come up; check the cell above")
print(f"endpoint up on :{PORT}")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = PUBLIC_URL + ROUTE
print()
print("Endpoint URL (paste into the Tool tab -> Configure endpoint):")
print("   ", ENDPOINT_URL)


In [ ]:
import json, urllib.error, urllib.request

# Built from the tool interface the Hub publishes for @examples/pvlib-python; edit `arguments` freely.
arguments = {"latitude": 37.0, "longitude": -122.0, "date": "2026-06-21"}
envelope = {"powerai": "1", "tool": TOOL_NAME, "input": arguments}

request = urllib.request.Request(
    ENDPOINT_URL, data=json.dumps(envelope).encode(),
    headers={"Content-Type": "application/json"},
)
reply = json.load(urllib.request.urlopen(request, timeout=90))
print(json.dumps(reply, indent=1))
assert reply.get("ok") is True, reply

# And that a bad request comes back as ok:false rather than as something a caller
# could mistake for a result — the field an agent platform actually reads.
broken = dict(envelope, input={k: v for k, v in arguments.items() if k != "latitude"})
try:
    urllib.request.urlopen(urllib.request.Request(
        ENDPOINT_URL, data=json.dumps(broken).encode(),
        headers={"Content-Type": "application/json"}), timeout=30)
    raise AssertionError("a request missing a required field should not succeed")
except urllib.error.HTTPError as err:
    failure = json.load(err)
    assert failure["ok"] is False, failure
    print("error path ->", failure["error"]["message"])

print()
print("endpoint verified")


### Register it, then clean up

Paste the URL above into the [Tool tab](https://hub.powerai.ai/examples/pvlib-python?tab=tool) → *Configure endpoint*. The
primitive then reports `invocable: true`, appears in `?invocable=true`, and is rendered
as an action in `/api/integrations/action-registry/?as=markdown` for an agent platform
to pull.

**When you finish, clear it** — the catalogue should not advertise an address that died
with this session:

```
curl -X PATCH <hub-api>/api/artifacts/examples/pvlib-python/endpoint/ \
  -H 'Content-Type: application/json' -d '{"tool_endpoint": null}'
```


## Your turn

`@examples/pvlib-python` is unpacked in `/content/primitive` and it is the working directory. Add a
cell and use it.

Nothing here is sent back to the Hub — this session is yours, and it disappears when
you close it. Colab's free tier gives you CPU always and a GPU when one is spare.
